# 02 — Pose Baselines and Latency Diagnostics

This notebook evaluates pose estimation and person detection candidates against the fixed `development` protocol, recording real wall-clock latency (p50, p90, p95, min, max, FPS throughput), person detection coverage, pose landmark coverage, and landmark confidence distributions on the local hardware.

It imports reusable evaluation functions from `kinetiq_v_vision.evaluation.baselines` and asserts benchmark invariants programmatically. In accordance with project policy, no metrics are fabricated and no bulky execution outputs or raw weights are committed to version control.

In [ ]:
import os
from pathlib import Path

from kinetiq_v_vision.evaluation.baselines import run_pose_baseline_benchmark

# Resolve manifest path from environment or fall back to verified synthetic CI fixture
DEFAULT_FIXTURE = Path("../fixtures/data/synthetic_manifest.json").resolve()
manifest_path = os.getenv("DATASET_MANIFEST_PATH", str(DEFAULT_FIXTURE))
split = os.getenv("BENCHMARK_SPLIT", "development")
print(f"Benchmarking pose candidates on '{split}' split from: {manifest_path}")

In [ ]:
report = run_pose_baseline_benchmark(
    manifest_path=manifest_path,
    split=split,
    max_frames_per_clip=10,
    warmup_frames=2,
)
print(report.summary_markdown())

In [ ]:
# Programmatic verification of benchmark invariants
assert report.total_clips_evaluated > 0, "No clips evaluated in benchmark"
assert len(report.candidate_metrics) >= 1, "No candidates evaluated"

for cid, metrics in report.candidate_metrics.items():
    assert metrics.total_frames > 0, f"Candidate {cid} evaluated zero frames"
    assert metrics.person_coverage_pct > 0.0, f"Candidate {cid} has zero person detection coverage"
    assert metrics.pose_coverage_pct > 0.0, f"Candidate {cid} has zero pose landmark coverage"
    assert metrics.p50_latency_ms > 0.0, f"Candidate {cid} p50 latency must be strictly positive"
    assert metrics.p95_latency_ms >= metrics.p50_latency_ms, f"Candidate {cid} p95 must be >= p50"
    assert metrics.throughput_fps > 0.0, f"Candidate {cid} throughput FPS must be positive"
    assert 0.0 <= metrics.mean_landmark_confidence <= 1.0, f"Candidate {cid} confidence out of bounds"

print("Pose baselines benchmark: all invariants verified.")